# Assign membrane objects to nuclei — **fast, vectorized notebook**
This notebook assigns nuclei objects to a membrane object with a single vectorized pass that counts co-occurrences of `(membrane_label, nucleus_label)` across voxels, then selects the **max-overlap nucleus per membrane** and finally enforces **one membrane per nucleus** by keeping the maximum-overlap pair per `nuc_label`.

**What you get:**
- A dataframe with output columns: `mem_label`, `nuc_label`, `Overlap Voxels`, `t`
- Clean batch runner with optional single-write or incremental append to CSV


In [ ]:
import os
import time
import numpy as np
import pandas as pd
from tifffile import imread


## Core function — vectorized overlap counting
Counts all `(mem_label, nuc_label)` pairs in one pass and keeps the best nucleus **per membrane**.

In [ ]:
def assign_nuclei_to_cells_vectorized(seg_membrane: np.ndarray, seg_nuclei: np.ndarray) -> pd.DataFrame:
    """
    Vectorized version: compute counts of (membrane_label, nucleus_label) co-occurrences
    and return, for each membrane label, the nucleus label with maximum overlap.
    Output columns: mem_label, nuc_label, Overlap Voxels
    """
    if seg_membrane.shape != seg_nuclei.shape:
        raise ValueError(f'Shape mismatch: membrane {seg_membrane.shape} vs nuclei {seg_nuclei.shape}')

    # Optionally narrow dtypes to save memory (uncomment if safe for your label ranges)
    # seg_membrane = seg_membrane.astype(np.uint32, copy=False)
    # seg_nuclei   = seg_nuclei.astype(np.uint32, copy=False)

    mem = seg_membrane.ravel()
    nuc = seg_nuclei.ravel()

    mask = (mem != 0) & (nuc != 0)
    if not np.any(mask):
        return pd.DataFrame(columns=['mem_label', 'nuc_label', 'Overlap Voxels'])

    mem_nz = mem[mask]
    nuc_nz = nuc[mask]

    # Unique (mem_label, nuc_label) pairs with counts
    pairs = np.stack([mem_nz, nuc_nz], axis=1)
    dtype = np.dtype([('m', mem_nz.dtype), ('n', nuc_nz.dtype)])
    pairs_view = pairs.view(dtype)
    uniq_pairs, counts = np.unique(pairs_view, return_counts=True)

    df = pd.DataFrame({
        'mem_label': uniq_pairs['m'],
        'nuc_label': uniq_pairs['n'],
        'Overlap Voxels': counts.astype(np.int32),
    })

    # Keep the nucleus with maximum overlap for each membrane
    idx = df.groupby('mem_label')['Overlap Voxels'].idxmax()
    df_best = df.loc[idx].reset_index(drop=True)
    return df_best


## Batch driver — tidy and robust
- Reads per timepoint files
- Selects best nucleus per membrane (above)
- **Enforces single membrane per nucleus** (keeps max overlap per `nuc_label`)
- Adds column `t` and writes CSV either once at the end (fastest) or incrementally (lower RAM).
- Batch processing expects separate folders of segmented nuclei and membrane label files
- `mem_file` expected format in the function is = 'final_relabelled_Merged-{t}_cp_masks.tif'
- `nuc_file` expected format is = 'Merged-{t}.tif', where `t` is the timepoint that will be added to the dataframe

In [ ]:
def process_tiff_folder_fast(membrane_folder: str,
                             nuclei_folder: str,
                             t_start: int = 1,
                             t_end: int = 305,
                             write_csv_path: str | None = None,
                             write_mode: str = 'single'  # 'single' or 'append'
                             ) -> pd.DataFrame:
    """
    Fast batch processor:
    - Reads membrane & nuclei per timepoint
    - Computes vectorized overlaps
    - Keeps best nucleus per membrane, then keeps best membrane per nucleus
    - Adds 't' column
    - Returns concatenated DataFrame (and optionally writes CSV).
    """
    all_results = []
    total_start = time.perf_counter()

    for t in range(t_start, t_end + 1):
        mem_file = f'final_relabelled_Merged-{t}_cp_masks.tif'
        nuc_file = f'Merged-{t}.tif'
        mem_path = os.path.join(membrane_folder, mem_file)
        nuc_path = os.path.join(nuclei_folder, nuc_file)

        tp_start = time.perf_counter()

        if not (os.path.exists(mem_path) and os.path.exists(nuc_path)):
            missing = ['membrane' if not os.path.exists(mem_path) else None,
                       'nuclei' if not os.path.exists(nuc_path) else None]
            missing = ','.join([m for m in missing if m])
            print(f'[skip t={t}] missing: {missing}')
            continue

        membrane = imread(mem_path)
        nuclei   = imread(nuc_path)

        # Compute best nucleus per membrane (vectorized)
        df_best = assign_nuclei_to_cells_vectorized(membrane, nuclei)

        # Enforce unique nucleus -> keep the membrane with maximum overlap for each nuc_label
        if not df_best.empty:
            idx2 = df_best.groupby('nuc_label')['Overlap Voxels'].idxmax()
            df_best = df_best.loc[idx2].reset_index(drop=True)
            df_best['t'] = t
        else:
            df_best = pd.DataFrame(columns=['mem_label', 'nuc_label', 'Overlap Voxels', 't'])

        tp_time = time.perf_counter() - tp_start
        print(f'[t={t}] {len(df_best)} rows, {tp_time:.2f}s')

        if write_csv_path and write_mode == 'append':
            header = not os.path.exists(write_csv_path)
            df_best.to_csv(write_csv_path, mode='a', index=False, header=header)
        else:
            all_results.append(df_best)

    total_time = time.perf_counter() - total_start
    print(f'[batch] done in {total_time/60:.1f} min')

    final_df = (pd.concat(all_results, ignore_index=True)
                if all_results else
                pd.DataFrame(columns=['mem_label', 'nuc_label', 'Overlap Voxels', 't']))

    if write_csv_path and write_mode == 'single':
        final_df.to_csv(write_csv_path, index=False)

    return final_df


## Parameters

In [ ]:
# <-- Update these paths for your environment -->
membrane_folder = r'D:/Mari/2024-07-25-transition/split_nuclei_membrane_raw/split_cellpose_results_relabelled/'
nuclei_folder   = r'D:/Mari/2024-07-25-transition/split_nuclei_membrane_raw/VollSeg/StarDist/'
output_csv      = r'D:/Mari/2024-07-25-transition/mem_nuc_overlap_table.csv'

# Write mode: 'single' writes once at end (fastest, uses RAM).
#              'append' writes incrementally (lower RAM).
write_mode = 'single'

t_start, t_end = 1, 305


## Run batch

In [ ]:
final_df = process_tiff_folder_fast(
    membrane_folder, nuclei_folder,
    t_start=t_start, t_end=t_end,
    write_csv_path=output_csv, write_mode=write_mode
)
final_df.head()


In [ ]:
final_df

## Note

Use the mem_nuc_overlap_table.csv file to next combine dataframes based on nuclei measurements and membrane measurements
Notebook version cleaned with the help of Microsoft's CoPilot AI, run and tested by Mari Tolonen, 05-12-2025